# SkillGraph — Data Collection & Provenance Audit (`01_data_collection.ipynb`)

Audits all raw, processed, and annotation datasets in `D:\\Downloads\\SkillGraph`, enforcing strict separation between:
- **Synthetic Development Corpora** (`2,000` global synthetic postings, `22` technical benchmark postings)
- **Unconfirmed Human Candidate Pool** (`human_annotation_sample_50.json` — `PENDING` review)
- **Real-World Collection Targets** (`real_job_postings.csv`, `real_university_majors.csv`, `real_university_courses.csv`)

In [1]:
import json
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.ingest_real_jobs import RealJobPostingPipeline
from scripts.ingest_curriculum import CurriculumIngestionPipeline

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'
ANN_DIR = PROJECT_ROOT / 'data' / 'annotations'

inventory = [
    {'file': 'data/raw/synthetic_job_postings.csv', 'provenance': 'synthetic', 'rows': len(pd.read_csv(RAW_DIR / 'synthetic_job_postings.csv'))},
    {'file': 'data/raw/real_job_postings.csv', 'provenance': 'real_target', 'rows': len(pd.read_csv(RAW_DIR / 'real_job_postings.csv'))},
    {'file': 'data/raw/real_university_majors.csv', 'provenance': 'real_target', 'rows': len(pd.read_csv(RAW_DIR / 'real_university_majors.csv'))},
    {'file': 'data/raw/real_university_courses.csv', 'provenance': 'real_target', 'rows': len(pd.read_csv(RAW_DIR / 'real_university_courses.csv'))},
    {'file': 'data/processed/global_jobs_cleaned.csv', 'provenance': 'synthetic_cleaned', 'rows': len(pd.read_csv(PROC_DIR / 'global_jobs_cleaned.csv'))},
    {'file': 'data/processed/global_skill_taxonomy.csv', 'provenance': 'taxonomy_v1', 'rows': len(pd.read_csv(PROC_DIR / 'global_skill_taxonomy.csv'))},
    {'file': 'data/processed/global_occupation_mapping.csv', 'provenance': 'occupations_v1', 'rows': len(pd.read_csv(PROC_DIR / 'global_occupation_mapping.csv'))},
]
display(pd.DataFrame(inventory))

,file,provenance,rows
0,data/raw/synthetic_job_postings.csv,synthetic,2000
1,data/raw/real_job_postings.csv,real_target,0
2,data/raw/real_university_majors.csv,real_target,0
3,data/raw/real_university_courses.csv,real_target,0
4,data/processed/global_jobs_cleaned.csv,synthetic_cleaned,2000
5,data/processed/global_skill_taxonomy.csv,taxonomy_v1,230
6,data/processed/global_occupation_mapping.csv,occupations_v1,53


In [2]:
job_pipe = RealJobPostingPipeline()
real_jobs_audit = job_pipe.process_and_audit(job_pipe.load_records(RAW_DIR / 'real_job_postings.csv'))

curr_pipe = CurriculumIngestionPipeline()
real_curr_audit = curr_pipe.ingest(
    majors=curr_pipe.load_records(RAW_DIR / 'real_university_majors.csv'),
    courses=curr_pipe.load_records(RAW_DIR / 'real_university_courses.csv'),
)

ann_json = json.loads((ANN_DIR / 'human_annotation_sample_50.json').read_text(encoding='utf-8-sig'))
pending_records = sum(1 for r in ann_json['records'] if r.get('record_status') == 'PENDING')

print('Real Job Postings Status:', real_jobs_audit['status'], '| Valid rows:', real_jobs_audit['valid_unique_rows'])
print('Real Curriculum Status  : Majors=', real_curr_audit['majors_validated'], '| Courses=', real_curr_audit['courses_validated'])
print('Human Sample 50 Status  :', ann_json.get('evaluation_type'), '| Pending records:', f"{pending_records}/{len(ann_json['records'])}")

Real Job Postings Status: AWAITING_REAL_DATA | Valid rows: 0
Real Curriculum Status  : Majors= 0 | Courses= 0
Human Sample 50 Status  : unconfirmed_candidate_pool | Pending records: 50/50
